# Prior Model — Deployment

Trains the prior XGBoost model on the **last 6 seasons** (2019-20 through 2025-26) using the hyperparameters and features tuned in `prior_modeling.ipynb`. No validation or test split — this is the production model.

In [1]:
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
from xgboost import XGBClassifier

DATA_DIR = Path('data')
MODEL_DIR = Path('prior_models')
MODEL_DIR.mkdir(exist_ok=True)

## 1. Load tuned config

In [2]:
# Load hyperparameters and feature list from the tuned model
config = json.load(open(MODEL_DIR / 'config.json'))
FEATURE_COLS = config['feature_cols']
BEST_ITERATION = config['best_iteration']
HYPERPARAMS = config['hyperparameters']

print(f'Features: {len(FEATURE_COLS)}')
print(f'Best iteration from tuning: {BEST_ITERATION}')
print(f'Hyperparameters: {HYPERPARAMS}')

Features: 159
Best iteration from tuning: 508
Hyperparameters: {'n_estimators': 1000, 'learning_rate': 0.02, 'max_depth': 2, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_lambda': 8.0, 'min_child_weight': 15, 'early_stopping_rounds': 30}


## 2. Load & filter training data (last 6 seasons)

In [3]:
df = pd.read_csv(DATA_DIR / 'training_games4.csv')
df['game_date'] = pd.to_datetime(df['game_date'])

# Assign season start year: games before July belong to previous year's season
df['season'] = df['game_date'].apply(lambda d: d.year if d.month >= 7 else d.year - 1)

print(f'Total games: {len(df)}')
print(f'Seasons available: {sorted(df["season"].unique())}')
print(f'Date range: {df["game_date"].min().date()} to {df["game_date"].max().date()}')

Total games: 17816
Seasons available: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Date range: 2010-11-14 to 2026-05-26


/var/folders/qd/69pzjl_56vjdn7_xy8vbzwt00000gn/T/ipykernel_2548/582864321.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['season'] = df['game_date'].apply(lambda d: d.year if d.month >= 7 else d.year - 1)


In [4]:
# Last 6 seasons: 2019-20 through 2025-26
TRAIN_SEASONS = list(range(2020, 2026))
df_train = df[df['season'].isin(TRAIN_SEASONS)].copy()

print(f'Training on seasons: {TRAIN_SEASONS}')
print(f'Training games: {len(df_train)}')
print(f'Date range: {df_train["game_date"].min().date()} to {df_train["game_date"].max().date()}')
print(f'\nGames per season:')
print(df_train.groupby('season').size())

Training on seasons: [2020, 2021, 2022, 2023, 2024, 2025]
Training games: 6839
Date range: 2020-07-30 to 2026-05-26

Games per season:
season
2020    1128
2021    1174
2022    1163
2023    1163
2024    1051
2025    1160
dtype: int64


In [5]:
X_train = df_train[FEATURE_COLS]
y_train = (df_train['score_diff'] > 0).astype(int)

print(f'Feature matrix: {X_train.shape}')
print(f'Home win rate: {y_train.mean():.3f}')
print(f'Missing values: {X_train.isna().sum().sum()}')

Feature matrix: (6839, 159)
Home win rate: 0.553
Missing values: 0


## 3. Train deployment model

In [6]:
# Use the exact hyperparameters from tuning, with n_estimators set to the
# best iteration found during early stopping (no val set to stop on here)
model = XGBClassifier(
    n_estimators=BEST_ITERATION,
    learning_rate=HYPERPARAMS['learning_rate'],
    max_depth=HYPERPARAMS['max_depth'],
    subsample=HYPERPARAMS['subsample'],
    colsample_bytree=HYPERPARAMS['colsample_bytree'],
    reg_lambda=HYPERPARAMS['reg_lambda'],
    min_child_weight=HYPERPARAMS['min_child_weight'],
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print(f'Model trained: {model.n_estimators} trees')

Model trained: 508 trees


## 4. Sanity check on training data

In [7]:
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

y_proba = model.predict_proba(X_train)[:, 1]
y_pred = (y_proba > 0.5).astype(int)

print('Training set metrics (sanity check — NOT generalization):')
print(f'  Accuracy:    {accuracy_score(y_train, y_pred):.4f}')
print(f'  ROC-AUC:     {roc_auc_score(y_train, y_proba):.4f}')
print(f'  Brier Score: {brier_score_loss(y_train, y_proba):.4f}')
print(f'  Log Loss:    {log_loss(y_train, y_proba):.4f}')

Training set metrics (sanity check — NOT generalization):
  Accuracy:    0.6883
  ROC-AUC:     0.7540
  Brier Score: 0.2015
  Log Loss:    0.5884


## 5. Save deployment model

In [8]:
# Save model
deploy_path = MODEL_DIR / 'xgboost_prior_deploy.pkl'
with open(deploy_path, 'wb') as f:
    pickle.dump(model, f)
print(f'Saved deployment model to {deploy_path}')

# Save config
deploy_config = {
    'model_type': 'XGBClassifier',
    'n_estimators': model.n_estimators,
    'hyperparameters': HYPERPARAMS,
    'feature_cols': FEATURE_COLS,
    'train_seasons': TRAIN_SEASONS,
    'train_games': len(df_train),
    'train_date_range': [str(df_train['game_date'].min().date()), str(df_train['game_date'].max().date())],
}

with open(MODEL_DIR / 'config_deploy.json', 'w') as f:
    json.dump(deploy_config, f, indent=2)
print(f'Saved deployment config to {MODEL_DIR / "config_deploy.json"}')

Saved deployment model to prior_models/xgboost_prior_deploy.pkl
Saved deployment config to prior_models/config_deploy.json


## 6. Generate predictions for all games

In [9]:
# Predict on ALL training games (for downstream posterior pipeline)
all_proba = model.predict_proba(X_train)[:, 1]

predictions_df = pd.DataFrame({
    'game_id': df_train['game_id'].values,
    'game_date': df_train['game_date'].values,
    'home_team': df_train['home_team'].values,
    'away_team': df_train['away_team'].values,
    'prior_home_wp': np.round(all_proba, 6),
    'home_win_actual': y_train.values,
    'score_diff': df_train['score_diff'].values if 'score_diff' in df_train.columns else 0,
})

predictions_df.to_csv(DATA_DIR / 'games_predictions_deploy.csv', index=False)
print(f'Saved predictions to data/games_predictions_deploy.csv ({len(predictions_df):,} games)')
print(predictions_df.tail(10).to_string(index=False))

Saved predictions to data/games_predictions_deploy.csv (6,839 games)
  game_id  game_date home_team away_team  prior_home_wp  home_win_actual  score_diff
401871339 2026-05-17       DET       CLE       0.676542                0         -31
401873197 2026-05-18       OKC        SA       0.615425                0          -7
401873341 2026-05-19        NY       CLE       0.703919                1          11
401873198 2026-05-20       OKC        SA       0.613023                1           9
401873342 2026-05-21        NY       CLE       0.735793                1          16
401873199 2026-05-22        SA       OKC       0.560419                0         -15
401873343 2026-05-23       CLE        NY       0.408773                0         -13
401873200 2026-05-24        SA       OKC       0.585413                1          21
401873344 2026-05-25       CLE        NY       0.410841                0         -37
401873201 2026-05-26       OKC        SA       0.573087                1         